In [1]:
import argparse
import os
import time
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

import torchvision
import torchvision.transforms as transforms

from models import *

global best_prec
use_gpu = torch.cuda.is_available()
print('=> Building model...')


batch_size = 128
model_name = "VGG16_quant_8x8"
model = VGG16_quant_8x8()
print(model)

normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])


train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ]))
trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)


test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ]))

testloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


print_freq = 100 # every 100 batches, accuracy printed. Here, each batch includes "batch_size" data points
# CIFAR10 has 50,000 training data, and 10,000 validation data.

def train(trainloader, model, criterion, optimizer, epoch):
    batch_time = AverageMeter()
    data_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    model.train()

    end = time.time()
    for i, (input, target) in enumerate(trainloader):
        # measure data loading time
        data_time.update(time.time() - end)

        input, target = input.cuda(), target.cuda()

        # compute output
        output = model(input)
        loss = criterion(output, target)

        # measure accuracy and record loss
        prec = accuracy(output, target)[0]
        losses.update(loss.item(), input.size(0))
        top1.update(prec.item(), input.size(0))

        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # measure elapsed time
        batch_time.update(time.time() - end)
        end = time.time()


        if i % print_freq == 0:
            print('Epoch: [{0}][{1}/{2}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   epoch, i, len(trainloader), batch_time=batch_time,
                   data_time=data_time, loss=losses, top1=top1))



def validate(val_loader, model, criterion ):
    batch_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    # switch to evaluate mode
    model.eval()

    end = time.time()
    with torch.no_grad():
        for i, (input, target) in enumerate(val_loader):

            input, target = input.cuda(), target.cuda()

            # compute output
            output = model(input)
            loss = criterion(output, target)

            # measure accuracy and record loss
            prec = accuracy(output, target)[0]
            losses.update(loss.item(), input.size(0))
            top1.update(prec.item(), input.size(0))

            # measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            if i % print_freq == 0:  # This line shows how frequently print out the status. e.g., i%5 => every 5 batch, prints out
                print('Test: [{0}/{1}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   i, len(val_loader), batch_time=batch_time, loss=losses,
                   top1=top1))

    print(' * Prec {top1.avg:.3f}% '.format(top1=top1))
    return top1.avg


def accuracy(output, target, topk=(1,)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk)
    batch_size = target.size(0)

    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target.view(1, -1).expand_as(pred))

    res = []
    for k in topk:
        correct_k = correct[:k].view(-1).float().sum(0)
        res.append(correct_k.mul_(100.0 / batch_size))
    return res


class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


def save_checkpoint(state, is_best, fdir):
    filepath = os.path.join(fdir, 'checkpoint.pth')
    torch.save(state, filepath)
    if is_best:
        shutil.copyfile(filepath, os.path.join(fdir, 'model_best.pth.tar'))


def adjust_learning_rate(optimizer, epoch):
    adjust_list = [20, 40]
    if epoch in adjust_list:
        for param_group in optimizer.param_groups:
            param_group['lr'] = param_group['lr'] * 0.1

#model = nn.DataParallel(model).cuda()
#all_params = checkpoint['state_dict']
#model.load_state_dict(all_params, strict=False)
#criterion = nn.CrossEntropyLoss().cuda()
#validate(testloader, model, criterion)

=> Building model...
VGG_quant(
  (features): Sequential(
    (0): QuantConv2d(
      3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
      (weight_quant): weight_quantize_fn()
    )
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): QuantConv2d(
      64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
      (weight_quant): weight_quantize_fn()
    )
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): QuantConv2d(
      64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
      (weight_quant): weight_quantize_fn()
    )
    (8): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU(inplace=True)
    (10): QuantConv2d(
      128, 128, kernel_size=(3, 3), stride

In [16]:
print(model.features[24:30])

Sequential(
  (24): QuantConv2d(
    256, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
    (weight_quant): weight_quantize_fn()
  )
  (25): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (26): ReLU(inplace=True)
  (27): QuantConv2d(
    8, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
    (weight_quant): weight_quantize_fn()
  )
  (28): ReLU(inplace=True)
  (29): QuantConv2d(
    8, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
    (weight_quant): weight_quantize_fn()
  )
)


In [17]:
# TRAIN VGG16 QUANT MODEL

# Hyperparameters
lr = 0.02
weight_decay = 5e-4
epochs = 100


best_prec = 0

#model = nn.DataParallel(model).cuda()
model.cuda()
criterion = nn.CrossEntropyLoss().cuda()
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
#cudnn.benchmark = True

if not os.path.exists('result'):
    os.makedirs('result')
fdir = 'result/'
if not os.path.exists(fdir):
    os.makedirs(fdir)


for epoch in range(0, epochs):
    train(trainloader, model, criterion, optimizer, epoch)

    # evaluate on test set
    print("Validation starts")
    prec = validate(testloader, model, criterion)
    scheduler.step()

    # remember best precision and save checkpoint
    is_best = prec > best_prec
    best_prec = max(prec,best_prec)
    print('best acc: {:1f}'.format(best_prec))
    save_checkpoint({
        'epoch': epoch + 1,
        'state_dict': model.state_dict(),
        'best_prec': best_prec,
        'optimizer': optimizer.state_dict(),
    }, is_best, fdir)

Epoch: [0][0/391]	Time 5.160 (5.160)	Data 5.142 (5.142)	Loss 2.6538 (2.6538)	Prec 7.812% (7.812%)
Epoch: [0][100/391]	Time 0.026 (0.077)	Data 0.000 (0.051)	Loss 2.3660 (2.9416)	Prec 8.594% (10.837%)
Epoch: [0][200/391]	Time 0.027 (0.052)	Data 0.001 (0.026)	Loss 2.2443 (2.6198)	Prec 14.844% (11.342%)
Epoch: [0][300/391]	Time 0.025 (0.043)	Data 0.001 (0.018)	Loss 2.0422 (2.4658)	Prec 24.219% (13.190%)
Validation starts
Test: [0/79]	Time 4.766 (4.766)	Loss 1.9175 (1.9175)	Prec 25.000% (25.000%)
 * Prec 26.680% 
best acc: 26.680000
Epoch: [1][0/391]	Time 5.066 (5.066)	Data 5.049 (5.049)	Loss 1.9136 (1.9136)	Prec 31.250% (31.250%)
Epoch: [1][100/391]	Time 0.026 (0.076)	Data 0.001 (0.050)	Loss 1.6630 (1.9012)	Prec 39.844% (26.771%)
Epoch: [1][200/391]	Time 0.025 (0.051)	Data 0.000 (0.026)	Loss 1.7778 (1.8573)	Prec 24.219% (28.113%)
Epoch: [1][300/391]	Time 0.026 (0.043)	Data 0.001 (0.017)	Loss 1.6589 (1.8192)	Prec 34.375% (29.288%)
Validation starts
Test: [0/79]	Time 4.776 (4.776)	Loss 1.559

In [2]:
USE_GPU = False  # True = GPU, False = CPU

FINAL_PATH = "final_trained_model/VGG16_quant_8x8_92.45_model.pth"
TRAIN_PATH = "result/model_best.pth.tar"

PATH = FINAL_PATH if os.path.exists(FINAL_PATH) else TRAIN_PATH
print("Loading:", PATH)

# Select device
device = torch.device("cuda" if USE_GPU and torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Load checkpoint onto selected device
checkpoint = torch.load(PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['state_dict'])

# Move model to selected device
model = model.to(device)
model.eval()

test_loss = 0
correct = 0

with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)
        correct += pred.eq(target.view_as(pred)).sum().item()

print('\nTest set: Accuracy: {}/{} ({:.2f}%)\n'.format(
    correct, len(testloader.dataset),
    100. * correct / len(testloader.dataset)
))


Loading: final_trained_model/VGG16_quant_8x8_92.45_model.pth
Using device: cpu

Test set: Accuracy: 9236/10000 (92.36%)



In [3]:
class SaveOutput:
    def __init__(self):
        self.outputs = []
    def __call__(self, module, module_in):
        self.outputs.append(module_in)
    def clear(self):
        self.outputs = []

######### Save input from 3rd layer ##########
save_8x8_in = SaveOutput()
save_8x8_out = SaveOutput()

# Extract the 3rd layer of VGG
h1 = model.features[27].register_forward_pre_hook(save_8x8_in)
h2 = model.features[29].register_forward_pre_hook(save_8x8_out)

dataiter = iter(testloader)
images, labels = next(dataiter)
images = images.to(device)
with torch.no_grad():
    out = model(images)

h1.remove()
h2.remove()

x = save_8x8_in.outputs[0][0][0] # single image
next_in = save_8x8_out.outputs[0][0][0]

#### Convert x into x_int ####
x_bit = 4

x_alpha  = model.features[27].act_alpha
x_delta = x_alpha / (2 ** x_bit - 1)

act_quant_fn = act_quantization(x_bit)
x_q = act_quant_fn(x, x_alpha)

x_int = (x_q / x_delta).round()
print("x_int shape:", x_int.shape)
print(x_int)

x_int shape: torch.Size([8, 4, 4])
tensor([[[ 0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  1.],
         [ 0.,  1.,  0.,  1.]],

        [[ 5.,  9., 11.,  6.],
         [ 5.,  6., 10.,  9.],
         [ 0.,  0.,  5., 14.],
         [ 0.,  0.,  0.,  2.]],

        [[ 0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  1.],
         [ 1.,  1.,  2.,  6.]],

        [[ 0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.],
         [ 2.,  0.,  0.,  0.]],

        [[ 0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  3.],
         [ 3.,  5.,  4.,  3.]],

        [[ 0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.],
         [11., 11.,  0.,  0.],
         [ 6.,  5.,  0.,  0.]],

        [[ 0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.]],

        [[ 6.,  8.,  5.,  2.],
         [ 5., 12.,  9.,  6.],
         [10., 15., 1

In [4]:
weight_q = model.features[27].weight_q
w_alpha = model.features[27].weight_quant.wgt_alpha
w_bit = 4

w_delta = w_alpha / (2**(w_bit-1) - 1) 
w_int = (weight_q / w_delta).round() # Round to fix floating point error
print("w_int shape:", w_int.shape)
print(w_int)

w_int shape: torch.Size([8, 8, 3, 3])
tensor([[[[ 7., -7., -7.],
          [-7.,  7., -7.],
          [ 7.,  7.,  7.]],

         [[ 7.,  7., -7.],
          [-7.,  7., -7.],
          [ 7., -7., -7.]],

         [[-7., -7.,  7.],
          [ 7., -7.,  7.],
          [ 7.,  7., -7.]],

         [[-7., -7., -7.],
          [-7., -7., -7.],
          [ 7.,  7., -7.]],

         [[ 7., -7.,  7.],
          [ 7.,  7., -7.],
          [ 7.,  7.,  7.]],

         [[ 7., -7.,  7.],
          [-7.,  7.,  7.],
          [-7.,  7.,  7.]],

         [[ 7.,  7.,  7.],
          [-1.,  7.,  5.],
          [-7., -7.,  7.]],

         [[ 7.,  7.,  7.],
          [ 7.,  7., -7.],
          [ 7., -7.,  7.]]],


        [[[-7., -7., -7.],
          [ 7., -7.,  7.],
          [ 7., -7., -7.]],

         [[-7.,  7.,  7.],
          [-7., -7.,  7.],
          [-7., -7.,  7.]],

         [[ 7.,  7., -7.],
          [ 7.,  7.,  7.],
          [ 7.,  7.,  7.]],

         [[ 7.,  7.,  7.],
          [ 7.,  7.,

In [5]:
conv_int = torch.nn.Conv2d(in_channels=8, out_channels=8, kernel_size=3, stride=1, padding=1, bias=False).to(device)
conv_int.weight = torch.nn.Parameter(w_int)

with torch.no_grad():
    psum_int = conv_int(x_int.unsqueeze(0))[0] # integer conv, [8, 4, 4]
    psum_recovered = psum_int * x_delta * w_delta # rescale to real units
    psum_recovered = F.relu(psum_recovered) # features[28] is a ReLU

difference = abs(psum_recovered - next_in)
print(difference.mean())
print("psum_recovered shape:", psum_recovered.shape)


tensor(3.7197e-08)
psum_recovered shape: torch.Size([8, 4, 4])


In [ ]:
ckpt = torch.load('result/VGG16_quant_8x8_92.45.pth.tar', map_location='cpu') # open the full file
torch.save({'state_dict': ckpt['state_dict'], # keep the weights
            'best_prec':  ckpt['best_prec'],
            'epoch':      ckpt['epoch']},
           'result/VGG16_quant_8x8_92.45_model.pth') # save without 'optimizer'

In [8]:
######### Create input.txt #########

# Pad input and flatten nij
x_padded = F.pad(x_int, (1, 1, 1, 1), value=0)
X = x_padded.reshape(8, 36)

print(X.shape)

bit_precision = 4
file = open('input.txt', 'w') #write to file
file.write('#time0row7[msb-lsb],time0row6[msb-lst],....,time0row0[msb-lst]#\n')
file.write('#time1row7[msb-lsb],time1row6[msb-lst],....,time1row0[msb-lst]#\n')
file.write('#................#\n')

for i in range(X.size(1)):  # time step
    for j in range(X.size(0)): # row #
        X_bin = '{0:04b}'.format(round(X[7-j,i].item()))
        for k in range(bit_precision):
            file.write(X_bin[k])        
        #file.write(' ')  # for visibility with blank between words, you can use
    file.write('\n')
file.close() #close file    


In [9]:
########## Create weight.txt ##########

# W = [kij, oc, ic]
W = w_int.reshape(8, 8, 9).permute(2, 0, 1)

bit_precision = 4
file = open('weight.txt', 'w')
file.write('#col0row7[msb-lsb],...,col0row0[msb-lsb]#\n')
file.write('#col1row7[msb-lsb],...,col1row0[msb-lsb]#\n')
file.write('#................#\n')

for kij in range(W.size(0)):
    for oc in range(W.size(1)):
        for j in range(W.size(2)):
            weight = round(W[kij, oc, 7-j].item())
            W_bin = '{0:04b}'.format(weight % 16)
            for k in range(bit_precision):
                file.write(W_bin[k])
        file.write('\n')
file.close()

In [15]:
########## Create psum.txt ##########

# P = [oc, onij] and apply ReLU
P = F.relu(psum_int).reshape(8, 16)

bit_precision = 16
file = open('psum.txt', 'w') #write to file
file.write('#time0col7[msb-lsb],time0col6[msb-lst],....,time0col0[msb-lst]#\n')
file.write('#time1col7[msb-lsb],time1col6[msb-lst],....,time1col0[msb-lst]#\n')
file.write('#................#\n')
for i in range(P.size(1)):
    for j in range(P.size(0)):
        value = round(P[7-j, i].item())

        # Convert to signed 16-bit two's complement
        P_bin = '{0:016b}'.format(value % (2**bit_precision))

        for k in range(bit_precision):
            file.write(P_bin[k])

    file.write('\n')

file.close()
